# Construir Aplicações de Geração de Imagens (OpenAI)

Há mais nos LLMs do que a geração de texto. Também pode gerar imagens a partir de descrições em texto. As imagens, enquanto modalidade, são úteis em áreas como MedTech, arquitetura, turismo, desenvolvimento de jogos, marketing, e mais. Nesta lição trabalhamos com os modelos **GPT Image** atuais na plataforma OpenAI.

## Objetivos de aprendizagem

No final desta lição será capaz de:

- Explicar o que é geração de imagens e onde é útil.
- Compreender a família de modelos `gpt-image` e como ela difere dos modelos legacy DALL·E.
- Construir uma aplicação de geração de imagens e editar imagens.

## O que é geração de imagens?

Os modelos de geração de imagens criam imagens a partir de um prompt em texto. Modelos modernos como o `gpt-image` aprendem a relação entre texto e imagens durante o treino, e depois convertem iterativamente ruído aleatório numa imagem que corresponde ao seu prompt.

Duas famílias bem conhecidas de modelos de imagem são:

- **`gpt-image` (OpenAI)** — a geração atual usada nesta lição. Suporta geração de imagem a partir de texto e edição de imagens (inpainting com máscara).
- **Midjourney** — um modelo popular de terceiros com o seu próprio serviço e fluxo de trabalho baseado em Discord.

> Os modelos de imagem mais antigos da OpenAI — **DALL·E 2** e **DALL·E 3** — são legados. O DALL·E 3 já não está disponível para novos deployments, e funcionalidades como `create_variation` só existiam no DALL·E 2. Use os modelos `gpt-image` para novas aplicações.

> **Importante:** os modelos `gpt-image` retornam a imagem gerada em **base64** (`b64_json`), não como URL. O seu código decodifica a string base64 para bytes e guarda-a — não existe URL da imagem para descarregar.


## Construir a sua primeira aplicação de geração de imagens

Então, o que é necessário para construir uma aplicação de geração de imagens? Precisa das seguintes bibliotecas:

- **python-dotenv**, é altamente recomendável usar esta biblioteca para manter os seus segredos num ficheiro *.env* separado do código.
- **openai**, esta biblioteca é aquela que vai usar para interagir com a API da OpenAI.
- **pillow**, para trabalhar com imagens em Python.


1. Crie um ficheiro *.env* com o seguinte conteúdo:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Reúna as bibliotecas acima num ficheiro chamado *requirements.txt* da seguinte forma:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. De seguida, crie um ambiente virtual e instale as bibliotecas:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Para Windows, utilize os seguintes comandos para criar e ativar o seu ambiente virtual:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Adicione o seguinte código no ficheiro chamado *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Criar objeto OpenAI (lê OPENAI_API_KEY do seu .env)
    client = openai.OpenAI()


    try:
        # Criar uma imagem utilizando a API de geração de imagens
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Introduza aqui o seu texto de prompt
            size='1024x1024',
            n=1
        )
        # Definir o diretório para a imagem armazenada
        image_dir = os.path.join(os.curdir, 'images')

        # Se o diretório não existir, criar
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inicializar o caminho da imagem (note que o tipo de ficheiro deve ser png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # modelos gpt-image devolvem a imagem em base64 (b64_json), não um URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Mostrar a imagem no visualizador de imagens predefinido
        image = Image.open(image_path)
        image.show()

    # capturar exceções
    except openai.BadRequestError as err:
        print(err)

    ```

Vamos explicar este código:

- Primeiro, importamos as bibliotecas que precisamos, incluindo a biblioteca OpenAI, a biblioteca dotenv, o módulo base64 e a biblioteca Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Depois disso, criamos o cliente, que lê a chave da API do seu ``.env``.

    ```python
    # Criar objeto OpenAI
    client = openai.OpenAI()
    ```

- De seguida, geramos a imagem:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Insira o seu texto de prompt aqui
        size='1024x1024',
        n=1
    )
    ```

    Os modelos `gpt-image` devolvem a imagem como uma string **base64** em `data[0].b64_json`. Decodificamo-la para bytes e escrevemo-la num ficheiro — não há URL para download.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Por último, abrimos a imagem e utilizamos o visualizador de imagens padrão para a mostrar:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Mais detalhes sobre a geração da imagem

Vamos analisar os parâmetros de `images.generate`:

- **model**, é o modelo de imagem a utilizar (por exemplo, `gpt-image-1`).
- **prompt**, é a descrição textual usada para gerar a imagem. Aqui é "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, é o tamanho da imagem gerada (por exemplo, `1024x1024`, `1536x1024`, `1024x1536`, ou `"auto"`).
- **n**, é o número de imagens geradas. Aqui geramos uma.

> Modelos de imagem não usam o parâmetro `temperature` — esse é um controlo para geração de texto. Para obter variedade, chame a API novamente; para reduzir a variedade, torne o seu prompt mais específico.

## Capacidades adicionais da geração de imagem

Já viu como gerar uma imagem com algumas linhas de Python. Os modelos `gpt-image` também podem **editar** uma imagem existente. Ao fornecer uma imagem, uma **máscara** opcional (que marca a área a alterar), e um prompt, pode modificar parte de uma imagem — por exemplo, adicionar um chapéu ao nosso coelho.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# as edições também são retornadas como base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

A imagem base contém apenas o coelho; a imagem final tem o chapéu no coelho.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
